# GPU Robot Lab: run every experiment

Before running: **Runtime, then Change runtime type**, pick a **T4 GPU**, save, then **Runtime, then Run all**. It takes about 15 to 25 minutes and ends by downloading **one zip file** with every result, chart and GIF. Send that zip (from your Downloads) back and the repo gets updated with real numbers.

If a cell fails, keep going: later cells still run. Mention which one failed.

In [ ]:
!nvidia-smi -L

In [ ]:
%cd /content
!rm -rf gpu-robot-lab
!git clone https://github.com/therealemmanueloppong/gpu-robot-lab.git
%cd gpu-robot-lab
!pip install -q mujoco mujoco-mjx
!pip install -q -e .
!mkdir -p logs

### Check that JAX can see the GPU
You want a `CudaDevice` below. If it says `CpuDevice`, the robot results will not show the GPU.

In [ ]:
import jax

print(jax.__version__, jax.devices())

## A. Matrix-multiply kernels

**A1. Are all kernels still correct?**

In [ ]:
!python scripts/check_kernels.py 2>&1 | tee logs/check_kernels.txt

**A2. How much do the results vary, and how do the versions compare?** Takes turns timing PyTorch, v1, v1 tuned and v2, ten times each.

In [ ]:
!python scripts/repeat_bench.py 2>&1 | tee logs/repeat_bench.txt

**A3. Which part of v2 helps?** Same tile settings, only the launch order changes (GROUP_M 1, 2, 4, 8, 16).

In [ ]:
!python scripts/ablate_grouping.py 2>&1 | tee logs/ablate_grouping.txt

**A4. Why is PyTorch still faster?** Does our kernel use the tensor cores (`mma` count), and what does PyTorch run?

In [ ]:
!python scripts/inspect_kernel.py 2>&1 | tee logs/inspect_kernel.txt
!python scripts/profile_torch.py 2>&1 | tee logs/profile_torch.txt

## B. Simulated robot

**B1. Simulation speed, cheap physics (a pendulum), repeated five times each.**

In [ ]:
!python scripts/bench_sim.py 2>&1 | tee logs/bench_sim_pendulum.txt

**B2. Simulation speed with collisions and friction (a box dropped on the floor).**

In [ ]:
!python scripts/bench_sim.py --model box --steps 300 --batches 1 16 256 1024 4096 2>&1 | tee logs/bench_sim_box.txt

**B3. Is the planner itself faster on a GPU? Same batch sizes, compile time excluded.**

In [ ]:
!JAX_PLATFORMS=cpu python scripts/bench_planner.py 2>&1 | tee logs/bench_planner_cpu.txt
!python scripts/bench_planner.py 2>&1 | tee logs/bench_planner_gpu.txt

**B4. Plan a swing-up with 4,096 parallel simulations.**

In [ ]:
!python scripts/run_swingup.py --batch 4096 --iterations 15 2>&1 | tee logs/run_swingup.txt

**B5. Train a neural-network policy with evolution strategies.** This is the learned controller.

In [ ]:
!python scripts/train_policy.py --population 4096 --iterations 150 2>&1 | tee logs/train_policy.txt

## C. Look at the results

In [ ]:
from IPython.display import Image, display

for f in ['results/swingup.gif', 'results/swingup_curve.png', 'results/policy/policy_swingup.gif', 'results/policy/policy_curve.png']:
    print(f)
    display(Image(f))

## D. Download everything as one zip

In [ ]:
!zip -q -r gpu_robot_lab_results.zip logs results *.csv
from google.colab import files

files.download('gpu_robot_lab_results.zip')